# AdaptiveDepth Phase 6.1 - response-preserving multi-route collection
This notebook evaluates Routes 20, 21, 22, and 23 against one reused full-model result on 400 fresh prompts. Use a T4 GPU. The full workspace is written directly to Drive and can resume after a runtime interruption.

In [ ]:
import base64, os, pathlib, shutil, subprocess, sys, zipfile

REPO_URL = 'https://github.com/VedantPandhare/AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs.git'
base = pathlib.Path('/content')
repo = base / 'AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs'

def read_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

if not repo.exists():
    clone_env = os.environ.copy()
    github_token = read_secret('GITHUB_TOKEN')
    if github_token:
        credential = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
        clone_env.update({'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.extraHeader', 'GIT_CONFIG_VALUE_0': f'Authorization: Basic {credential}'})
    clone = subprocess.run(['git', 'clone', REPO_URL, str(repo)], env=clone_env, capture_output=True, text=True)
    if clone.returncode:
        detail = clone.stderr.strip().splitlines()[-1] if clone.stderr.strip() else 'unknown Git error'
        raise RuntimeError(f'Git clone failed: {detail}\nFor a private repository, add a read-only GITHUB_TOKEN to Colab Secrets.')
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[data]'], check=True)
subprocess.run(['nvidia-smi'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before Phase 6.1'
print('Repository:', repo)
print('GPU:', torch.cuda.get_device_name(0))

## Mount Drive and stage the corrected checkpoint
The model is extracted locally for speed; only the small resumable experiment workspace is written to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_results = pathlib.Path('/content/drive/MyDrive/AdaptiveDepth-results')
archive = drive_results / 'phase5_corrected_student.zip'
extract_root = pathlib.Path('/content/phase5_corrected_for_phase61')
if not archive.is_file():
    raise FileNotFoundError(f'Missing corrected checkpoint archive: {archive}')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(archive) as bundle:
    bundle.extractall(extract_root)
matches = [path.parent for path in extract_root.rglob('model.safetensors')]
if len(matches) != 1:
    raise RuntimeError(f'Expected one model checkpoint, found: {matches}')
checkpoint = matches[0]
required = ['model.safetensors', 'config.json', 'tokenizer.json', 'tokenizer_config.json']
missing = [name for name in required if not (checkpoint / name).is_file()]
if missing:
    raise RuntimeError(f'Incomplete corrected checkpoint: {missing}')
print('Checkpoint:', checkpoint)
print('Model size GB:', round((checkpoint / 'model.safetensors').stat().st_size / 1024**3, 3))

## Select smoke or the combined 400-prompt run
Run smoke once. Then set `RUN_FULL = True` and rerun this cell plus the final two cells. The full workspace persists on Drive and `--resume` skips completed prompts.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    dataset = pathlib.Path('data/phase61_calibration.jsonl')
    if not dataset.exists():
        subprocess.run([
            'adaptivedepth', 'prepare-phase61-data',
            '--output', str(dataset), '--max-examples', '400',
            '--skip-examples', '2500', '--seed', '17',
        ], check=True)
    config = 'configs/phase61_t4.yaml'
    workspace = drive_results / 'phase61_workspace'
else:
    dataset = pathlib.Path('data/phase6_calibration_smoke.jsonl')
    config = 'configs/phase61_smoke.yaml'
    workspace = pathlib.Path('experiments/phase61_smoke')
workspace.mkdir(parents=True, exist_ok=True)
print('Mode:', 'FULL' if RUN_FULL else 'SMOKE')
print('Dataset:', dataset, 'examples:', sum(1 for line in dataset.open(encoding='utf-8') if line.strip()))
print('Workspace:', workspace)

In [ ]:
subprocess.run([
    'adaptivedepth', 'run-phase61',
    '--checkpoint', str(checkpoint),
    '--config', config,
    '--routes-file', 'configs/phase61_routes.yaml',
    '--dataset', str(dataset),
    '--workspace', str(workspace),
    '--resume',
], check=True)

In [ ]:
print((workspace / 'gate.md').read_text())
output_name = 'phase61_artifacts.zip' if RUN_FULL else 'phase61_smoke_artifacts.zip'
output_zip = pathlib.Path('/content') / output_name
with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in workspace.rglob('*'):
        if path.is_file():
            bundle.write(path, 'phase61/' + str(path.relative_to(workspace)))
drive_results.mkdir(parents=True, exist_ok=True)
saved = shutil.copy2(output_zip, drive_results / output_zip.name)
shutil.copy2(workspace / 'gate.md', drive_results / ('phase61_gate.md' if RUN_FULL else 'phase61_smoke_gate.md'))
shutil.copy2(workspace / 'gate.json', drive_results / ('phase61_gate.json' if RUN_FULL else 'phase61_smoke_gate.json'))
print('Saved:', saved)
if not RUN_FULL:
    print('Smoke complete. Set RUN_FULL = True and rerun the final three code cells.')
else:
    print('Upload phase61_artifacts.zip or paste gate.md for Phase 7.1 planning.')